# 03 — Traditional Baselines: ARIMA & Random Forest
Train and evaluate non-deep-learning models as baselines.

These models do **NOT** use graph structure — they treat each sensor independently.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.config import set_seed
from src.data_loader import prepare_dataset
from src.evaluate import evaluate_predictions, print_results
set_seed(42)

In [2]:
# Prepare data
data = prepare_dataset(config.METR_LA_PATH, batch_size=config.BATCH_SIZE)
mean, std = data['mean'], data['std']
splits = data['splits']
num_sensors = splits['train'][0].shape[2]
test_Y = splits['test'][1]
all_results = {}
all_preds = {}

Loading /home/anonymous/GraphNN/dataset/METR-LA.csv...
  Shape: (34272, 207) (timesteps × sensors)
  Time range: 2012-03-01 00:00:00 to 2012-06-27 23:55:00
  Missing values: 0 (0.00%)
  Zero values: 575302 (8.11%)
Handling missing values...
  After cleaning — NaN: 0, Zeros: 0
Creating sequences...
  Created 34249 sequences: X=(34249, 12, 207), Y=(34249, 12, 207)
Splitting data...
  train: 23974 samples
  val: 3425 samples
  test: 6850 samples


## 1. Historical Average (Simplest Baseline)
Predict: mean of the input window → replicate for all horizons.

In [3]:
test_X = splits['test'][0]
ha_preds = np.mean(test_X, axis=1, keepdims=True)
ha_preds = np.repeat(ha_preds, config.PRED_LEN, axis=1)

r = evaluate_predictions(ha_preds, test_Y, mean, std)
print_results(r, 'Historical Average')
all_results['HistAvg'] = r
all_preds['HistAvg'] = ha_preds


  Results: Historical Average
  Horizon           MAE     RMSE  MAPE(%)
  ----------------------------------------
  15min          3.6208   7.2859     9.79
  30min          4.1987   8.4633    11.54
  60min          5.2008  10.2695    14.57
  overall        4.2521   8.5925    11.70



## 2. Last Value (Naive Baseline)
Predict: repeat the last observed value for all future steps.

In [4]:
lv_preds = np.repeat(test_X[:, -1:, :], config.PRED_LEN, axis=1)

r = evaluate_predictions(lv_preds, test_Y, mean, std)
print_results(r, 'Last Value')
all_results['LastValue'] = r
all_preds['LastValue'] = lv_preds


  Results: Last Value
  Horizon           MAE     RMSE  MAPE(%)
  ----------------------------------------
  15min          3.1598   6.1409     7.78
  30min          3.8279   7.6891     9.87
  60min          4.9021   9.8031    13.14
  overall        3.8538   7.7997     9.93



## 3. ARIMA(3,0,1)
Per-sensor univariate time series model.
Fits on training data, forecasts using input window as recent context.

**Runtime: ~10 minutes** (30 sensors sampled)

In [ ]:
from src.models.arima_model import ARIMAForecaster

# Reconstruct training time series
train_X = splits['train'][0]
T_train = len(train_X) + config.SEQ_LEN - 1
train_series = np.zeros((T_train, num_sensors))
train_series[:config.SEQ_LEN] = train_X[0]
for i in range(1, len(train_X)):
    train_series[config.SEQ_LEN + i - 1] = train_X[i, -1]

arima = ARIMAForecaster(order=config.ARIMA_ORDER, max_sensors=config.ARIMA_MAX_SENSORS)
preds = arima.fit_and_predict(train_series, splits['test'][0], config.PRED_LEN)

r = evaluate_predictions(preds, test_Y, mean, std)
print_results(r, arima.get_name())
all_results['ARIMA'] = r
all_preds['ARIMA'] = preds

## 4. Random Forest
Uses past 12 timesteps as features, one RF per horizon step.
Each sensor treated independently (flattened).

**Runtime: ~1 minute**

In [ ]:
from src.models.rf_model import RandomForestForecaster

rf = RandomForestForecaster(n_estimators=config.RF_N_ESTIMATORS,
                            max_depth=config.RF_MAX_DEPTH, n_jobs=config.RF_N_JOBS)
rf.fit(splits['train'][0], splits['train'][1])
preds = rf.predict(splits['test'][0])

r = evaluate_predictions(preds, test_Y, mean, std)
print_results(r, rf.get_name())
all_results['RandomForest'] = r
all_preds['RandomForest'] = preds

## 5. Comparison of All Traditional Baselines

In [ ]:
from src.evaluate import compare_models
compare_models(all_results, 'METR-LA')

In [ ]:
# Plot predictions for sensor 0
from src.visualize import plot_predictions
plot_predictions(all_preds, test_Y, mean, std, sensor_idx=0, time_range=(0, 288),
                 save_path='../results/plots/baselines_pred_sensor0.png')

# Results table
rows = []
for model_name, r in all_results.items():
    for h in ['15min', '30min', '60min']:
        if h in r:
            rows.append({'Model': model_name, 'Horizon': h,
                         'MAE': r[h]['MAE'], 'RMSE': r[h]['RMSE'], 'MAPE': r[h]['MAPE']})
df = pd.DataFrame(rows)
print(df.pivot_table(index='Model', columns='Horizon', values='MAE').to_string(float_format='%.4f'))

# Save
from src.evaluate import save_results
save_results(all_results, '../results/metrics', 'METR-LA_baselines')
print("\nBaseline results saved!")